# NILM Approach 2 — Appliance-Specific Specialist Models + Meta Model

**Updated experiment settings**
- Dataset: UK-DALE House 1 processed 2-year CSV
- Appliances: laptop, washing_machine, tv, fridge, microwave
- Window size: **100 samples (~10 minutes)**
- Stride: **20 samples (~2 minutes)**
- Label: **center-point appliance ON/OFF state**
- Split: chronological 70% train / 15% meta / 15% test
- Specialist models: one model per appliance
- Each specialist predicts **only its own appliance**
- Same CNN + BiLSTM backbone for all specialists
- **12 appliance-profile knowledge features** instead of the previous 13 generic statistical features
- Each specialist produces a **128-D learned feature + 1 probability**
- Meta input: **645 features = 640 learned features + 5 probabilities**
- Meta model: **645 → 256 → 128 → 5**
- Epochs: 10
- Batch size: 128

### Appliance-profile knowledge features
1. typical_power_on
2. delta_power_on_off
3. standby_power
4. power_std_on
5. typical_on_duration_sec
6. typical_off_duration_sec
7. dominant_cycle_period_sec
8. duty_cycle
9. operations_per_day
10. peak_hour_of_day
11. hour_concentration
12. weekday_weekend_ratio

**Important:** appliance channels are used only to build the training-time appliance profiles and labels. The model input remains aggregate household power + the specialist's fixed appliance-profile vector.

In [ ]:
# Cell 1 — Imports and Google Drive
import os, json, numpy as np, pandas as pd, tensorflow as tf

from google.colab import drive
drive.mount('/content/gdrive')

from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import (
    Input, Conv1D, BatchNormalization, MaxPooling1D,
    Bidirectional, LSTM, GlobalAveragePooling1D,
    Dense, Dropout, Concatenate
)
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

print("TensorFlow:", tf.__version__)

Mounted at /content/gdrive
TensorFlow: 2.20.0


In [ ]:
# Cell 2 — Configuration

DATA_PATH = "/content/gdrive/MyDrive/house_1/house_1_fully/house1_required_2years.csv"

SAVE_ROOT = "/content/gdrive/MyDrive/NILM_Approach2_2Years/improved_100_center"
BASE_DIR = os.path.join(SAVE_ROOT, "specialist_models")
META_DIR = os.path.join(SAVE_ROOT, "meta_model")

os.makedirs(BASE_DIR, exist_ok=True)
os.makedirs(META_DIR, exist_ok=True)

APPLIANCES = ["laptop", "washing_machine", "tv", "fridge", "microwave"]

WINDOW_SIZE = 100
STRIDE = 20
BATCH_SIZE = 128
SPECIALIST_EPOCHS = 10
META_EPOCHS = 10

THRESHOLD_W = 10.0

print("Window:", WINDOW_SIZE)
print("Stride:", STRIDE)
print("Specialists:", APPLIANCES)
print("Save directory:", SAVE_ROOT)

Window: 100
Stride: 20
Specialists: ['laptop', 'washing_machine', 'tv', 'fridge', 'microwave']
Save directory: /content/gdrive/MyDrive/NILM_Approach2_2Years/improved_100_center


In [ ]:
# Cell 3 — Load processed dataset

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())

Shape: (9469092, 8)
Columns: ['datetime', 'timestamp', 'aggregate', 'laptop', 'washing_machine', 'tv', 'fridge', 'microwave']


,datetime,timestamp,aggregate,laptop,washing_machine,tv,fridge,microwave
0,2012-11-09 22:28:15+00:00,1352500095,599,0.0,0.0,98.0,0.0,0.0
1,2012-11-09 22:28:21+00:00,1352500101,582,0.0,0.0,98.0,0.0,0.0
2,2012-11-09 22:28:27+00:00,1352500107,600,0.0,0.0,100.0,0.0,0.0
3,2012-11-09 22:28:33+00:00,1352500113,586,0.0,0.0,99.0,0.0,0.0
4,2012-11-09 22:28:40+00:00,1352500120,596,0.0,0.0,100.0,0.0,0.0


In [ ]:
# Cell 4 — Clean data and create appliance ON/OFF labels

required_cols = ["datetime", "timestamp", "aggregate"] + APPLIANCES
df = df[required_cols].copy()

for c in ["aggregate"] + APPLIANCES:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.dropna(subset=["aggregate"] + APPLIANCES).reset_index(drop=True)

Y_state = (df[APPLIANCES].values >= THRESHOLD_W).astype(np.float32)
aggregate = df["aggregate"].values.astype(np.float32)

print("Clean shape:", df.shape)
print("Y_state shape:", Y_state.shape)

for j, app in enumerate(APPLIANCES):
    off = int((Y_state[:, j] == 0).sum())
    on = int((Y_state[:, j] == 1).sum())
    print(f"{app:18s} OFF={off:,} ON={on:,} ON%={100*on/(off+on):.2f}")

Clean shape: (9469092, 8)
Y_state shape: (9469092, 5)
laptop             OFF=7,778,834 ON=1,690,258 ON%=17.85
washing_machine    OFF=9,051,622 ON=417,470 ON%=4.41
tv                 OFF=8,611,285 ON=857,807 ON%=9.06
fridge             OFF=5,731,340 ON=3,737,752 ON%=39.47
microwave          OFF=9,428,809 ON=40,283 ON%=0.43


In [ ]:
# Cell 5 — Appliance Profile Knowledge Features
#
# These are the 12 knowledge features you selected.
# They are calculated from the TRAIN portion of each appliance channel
# and then used as fixed appliance-profile knowledge for that specialist.
#
# IMPORTANT:
# The appliance channels are NOT model inputs.
# They are used only to build training-time appliance profiles.

PROFILE_FEATURE_NAMES = [
    # Power (4)
    "typical_power_on", "delta_power_on_off", "standby_power", "power_std_on",
    # Temporal (4)
    "typical_on_duration_sec", "typical_off_duration_sec",
    "dominant_cycle_period_sec", "duty_cycle",
    # Usage (4)
    "operations_per_day", "peak_hour_of_day", "hour_concentration",
    "weekday_weekend_ratio",
]

print("Profile knowledge features:", len(PROFILE_FEATURE_NAMES))
print(PROFILE_FEATURE_NAMES)

Profile knowledge features: 12
['typical_power_on', 'delta_power_on_off', 'standby_power', 'power_std_on', 'typical_on_duration_sec', 'typical_off_duration_sec', 'dominant_cycle_period_sec', 'duty_cycle', 'operations_per_day', 'peak_hour_of_day', 'hour_concentration', 'weekday_weekend_ratio']


In [ ]:
# Cell 6 — Build 100-sample windows with center-point labels

X_windows = []
Y_windows = []
centers = []

n = len(aggregate)

for start in range(0, n - WINDOW_SIZE + 1, STRIDE):
    end = start + WINDOW_SIZE
    center = start + WINDOW_SIZE // 2

    X_windows.append(aggregate[start:end])
    Y_windows.append(Y_state[center])
    centers.append(center)

X_windows = np.asarray(X_windows, dtype=np.float32)
Y_windows = np.asarray(Y_windows, dtype=np.float32)
centers = np.asarray(centers)

print("X_windows:", X_windows.shape)
print("Y_windows:", Y_windows.shape)

print("\nCenter-point label distribution:")
for j, app in enumerate(APPLIANCES):
    off = int((Y_windows[:, j] == 0).sum())
    on = int((Y_windows[:, j] == 1).sum())
    print(f"{app:18s} OFF={off:,} ON={on:,} ON%={100*on/(off+on):.2f}")

X_windows: (473450, 100)
Y_windows: (473450, 5)

Center-point label distribution:
laptop             OFF=389,016 ON=84,434 ON%=17.83
washing_machine    OFF=452,637 ON=20,813 ON%=4.40
tv                 OFF=430,523 ON=42,927 ON%=9.07
fridge             OFF=286,494 ON=186,956 ON%=39.49
microwave          OFF=471,416 ON=2,034 ON%=0.43


In [ ]:
# Cell 7 — Chronological 70 / 15 / 15 split

N = len(X_windows)

i1 = int(0.70 * N)
i2 = int(0.85 * N)

X_train, X_meta, X_test = X_windows[:i1], X_windows[i1:i2], X_windows[i2:]
Y_train, Y_meta, Y_test = Y_windows[:i1], Y_windows[i1:i2], Y_windows[i2:]

C_train, C_meta, C_test = centers[:i1], centers[i1:i2], centers[i2:]

print("Train:", X_train.shape, Y_train.shape)
print("Meta :", X_meta.shape, Y_meta.shape)
print("Test :", X_test.shape, Y_test.shape)

Train: (331415, 100) (331415, 5)
Meta : (71017, 100) (71017, 5)
Test : (71018, 100) (71018, 5)


In [ ]:
# Cell 8 — Standardize aggregate windows using TRAIN only

agg_scaler = StandardScaler()

X_train_s = agg_scaler.fit_transform(
    X_train.reshape(-1, 1)
).reshape(X_train.shape).astype(np.float32)

X_meta_s = agg_scaler.transform(
    X_meta.reshape(-1, 1)
).reshape(X_meta.shape).astype(np.float32)

X_test_s = agg_scaler.transform(
    X_test.reshape(-1, 1)
).reshape(X_test.shape).astype(np.float32)

print("Aggregate scaling complete.")

Aggregate scaling complete.


In [ ]:
# Cell 10 — Appliance-specific specialist architecture

PROFILE_DIM = len(PROFILE_FEATURE_NAMES)

def build_specialist_model():
    agg_in = Input(shape=(WINDOW_SIZE, 1), name="aggregate_input")
    profile_in = Input(shape=(PROFILE_DIM,), name="appliance_profile_input")

    x = Conv1D(32, 7, activation="relu")(agg_in)
    x = BatchNormalization()(x)
    x = MaxPooling1D(2)(x)

    x = Conv1D(64, 5, activation="relu")(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(2)(x)

    x = Bidirectional(LSTM(64, return_sequences=True))(x)
    x = GlobalAveragePooling1D()(x)

    k = Dense(64, activation="relu")(profile_in)
    k = Dropout(0.20)(k)

    fused = Concatenate()([x, k])
    fused = Dense(128, activation="relu")(fused)
    fused = Dropout(0.30)(fused)

    learned = Dense(128, activation="relu", name="learned_feature_128")(fused)
    output = Dense(1, activation="sigmoid", name="state_output")(learned)

    model = Model([agg_in, profile_in], output)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall")
        ]
    )

    feature_model = Model(
        inputs=model.inputs,
        outputs=model.get_layer("learned_feature_128").output
    )

    return model, feature_model

test_model, test_feature = build_specialist_model()
test_model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ aggregate_input     │ (None, 100, 1)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 94, 32)    │        256 │ aggregate_input[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 94, 32)    │        128 │ conv1d[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d       │ (None, 47, 32)    │          0 │ batch_normalizat… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 43, 64)    │     10,304 │ max_pooling1d[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 43, 64)    │        256 │ conv1d_1[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_1     │ (None, 21, 64)    │          0 │ batch_normalizat… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ appliance_profile_… │ (None, 12)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 21, 128)   │     66,048 │ max_pooling1d_1[… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 64)        │        832 │ appliance_profil… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 128)       │          0 │ bidirectional[0]… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 64)        │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 192)       │          0 │ global_average_p… │
│ (Concatenate)       │                   │            │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 128)       │     24,704 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 128)       │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ learned_feature_128 │ (None, 128)       │     16,512 │ dropout_1[0][0]   │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ state_output        │ (None, 1)         │        129 │ learned_feature_… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 119,169 (465.50 KB)

 Trainable params: 118,977 (464.75 KB)

 Non-trainable params: 192 (768.00 B)

In [ ]:
# ============================================================
# CELL 9 — CREATE THE 12 APPLIANCE PROFILE KNOWLEDGE FEATURES
# ============================================================

PROFILE_FEATURE_NAMES = [
    # Power (4)
    "typical_power_on",
    "delta_power_on_off",
    "standby_power",
    "power_std_on",

    # Temporal (4)
    "typical_on_duration_sec",
    "typical_off_duration_sec",
    "dominant_cycle_period_sec",
    "duty_cycle",

    # Usage (4)
    "operations_per_day",
    "peak_hour_of_day",
    "hour_concentration",
    "weekday_weekend_ratio",
]

print("Number of profile features:", len(PROFILE_FEATURE_NAMES))
print(PROFILE_FEATURE_NAMES)


# ------------------------------------------------------------
# Estimate sampling interval
# ------------------------------------------------------------

try:
    ts = pd.to_datetime(df["datetime"])

    sample_seconds = float(
        ts.diff()
        .dt.total_seconds()
        .median()
    )

    if not np.isfinite(sample_seconds) or sample_seconds <= 0:
        sample_seconds = 6.0

except Exception:
    sample_seconds = 6.0


print("\nEstimated sampling interval:",
      sample_seconds, "seconds")


# ============================================================
# FUNCTION TO CALCULATE ONE APPLIANCE PROFILE
# ============================================================

def calculate_profile(app, train_end_row):

    power = df[app].iloc[:train_end_row].to_numpy(
        dtype=np.float32
    )

    timestamps = pd.to_datetime(
        df["datetime"].iloc[:train_end_row]
    )

    # --------------------------------------------------------
    # ON / OFF
    # --------------------------------------------------------

    on = power >= THRESHOLD_W


    # ========================================================
    # 1. POWER FEATURES
    # ========================================================

    on_values = power[on]
    off_values = power[~on]

    # Typical ON power
    if len(on_values) > 0:
        typical_power_on = float(
            np.median(on_values)
        )
    else:
        typical_power_on = 0.0

    # Standby / OFF power
    if len(off_values) > 0:
        standby_power = float(
            np.median(off_values)
        )
    else:
        standby_power = 0.0

    # Difference between ON and OFF
    delta_power_on_off = (
        typical_power_on - standby_power
    )

    # Variation while ON
    if len(on_values) > 0:
        power_std_on = float(
            np.std(on_values)
        )
    else:
        power_std_on = 0.0


    # ========================================================
    # 2. TEMPORAL FEATURES
    # ========================================================

    binary = on.astype(np.int8)

    if len(binary) > 0:

        changes = np.r_[
            True,
            binary[1:] != binary[:-1]
        ]

        starts = np.flatnonzero(changes)

        ends = np.r_[
            starts[1:],
            len(binary)
        ]

        values = binary[starts]

        on_runs = (
            ends[values == 1]
            -
            starts[values == 1]
        )

        off_runs = (
            ends[values == 0]
            -
            starts[values == 0]
        )

    else:

        on_runs = np.array([])
        off_runs = np.array([])


    # Typical ON duration
    if len(on_runs) > 0:

        typical_on_duration_sec = float(
            np.median(on_runs)
            * sample_seconds
        )

    else:

        typical_on_duration_sec = 0.0


    # Typical OFF duration
    if len(off_runs) > 0:

        typical_off_duration_sec = float(
            np.median(off_runs)
            * sample_seconds
        )

    else:

        typical_off_duration_sec = 0.0


    # --------------------------------------------------------
    # Dominant cycle period
    # --------------------------------------------------------

    on_starts = np.flatnonzero(
        on &
        ~np.r_[
            False,
            on[:-1]
        ]
    )

    if len(on_starts) >= 2:

        cycle_diffs = (
            np.diff(on_starts)
            * sample_seconds
        )

        dominant_cycle_period_sec = float(
            np.median(cycle_diffs)
        )

    else:

        dominant_cycle_period_sec = 0.0


    # Duty cycle
    if len(on) > 0:

        duty_cycle = float(
            np.mean(on)
        )

    else:

        duty_cycle = 0.0


    # ========================================================
    # 3. USAGE FEATURES
    # ========================================================

    dates = timestamps.dt.date.to_numpy()
    hours = timestamps.dt.hour.to_numpy()
    weekdays = timestamps.dt.dayofweek.to_numpy()


    # --------------------------------------------------------
    # Operations per day
    # --------------------------------------------------------

    unique_days = np.unique(dates)

    operations_each_day = []

    for day in unique_days:

        mask = dates == day

        day_on = on[mask]

        if len(day_on) == 0:
            continue

        day_starts = np.sum(
            day_on &
            ~np.r_[
                False,
                day_on[:-1]
            ]
        )

        operations_each_day.append(
            int(day_starts)
        )


    if len(operations_each_day) > 0:

        operations_per_day = float(
            np.mean(operations_each_day)
        )

    else:

        operations_per_day = 0.0


    # --------------------------------------------------------
    # Peak hour of day
    # --------------------------------------------------------

    if np.any(on):

        hourly_on = np.bincount(
            hours[on],
            minlength=24
        )

        peak_hour_of_day = float(
            np.argmax(hourly_on)
        )

    else:

        peak_hour_of_day = 0.0


    # --------------------------------------------------------
    # Hour concentration
    # --------------------------------------------------------

    if np.any(on):

        hourly_on = np.bincount(
            hours[on],
            minlength=24
        ).astype(float)

        total_hourly = np.bincount(
            hours,
            minlength=24
        ).astype(float)

        hourly_rate = (
            hourly_on /
            np.maximum(
                total_hourly,
                1.0
            )
        )

        hour_concentration = float(
            np.max(hourly_rate)
        )

    else:

        hour_concentration = 0.0


    # --------------------------------------------------------
    # Weekday / weekend ratio
    # --------------------------------------------------------

    weekday_mask = weekdays < 5
    weekend_mask = weekdays >= 5

    weekday_on = np.sum(
        on[weekday_mask]
    )

    weekend_on = np.sum(
        on[weekend_mask]
    )

    weekday_days = max(
        len(np.unique(dates[weekday_mask])),
        1
    )

    weekend_days = max(
        len(np.unique(dates[weekend_mask])),
        1
    )

    weekday_rate = (
        weekday_on /
        weekday_days
    )

    weekend_rate = (
        weekend_on /
        weekend_days
    )

    weekday_weekend_ratio = float(
        weekday_rate /
        max(weekend_rate, 1e-8)
    )


    # ========================================================
    # RETURN 12 FEATURES
    # ========================================================

    return np.array([

        typical_power_on,
        delta_power_on_off,
        standby_power,
        power_std_on,

        typical_on_duration_sec,
        typical_off_duration_sec,
        dominant_cycle_period_sec,
        duty_cycle,

        operations_per_day,
        peak_hour_of_day,
        hour_concentration,
        weekday_weekend_ratio

    ], dtype=np.float32)


# ============================================================
# CALCULATE PROFILES USING TRAINING DATA ONLY
# ============================================================

train_end_original_row = int(
    C_train[-1] + 1
)

print("\nTraining data ends at original row:",
      train_end_original_row)


PROFILE_MATRIX = np.vstack([

    calculate_profile(
        app,
        train_end_original_row
    )

    for app in APPLIANCES

])


# ============================================================
# DISPLAY PROFILE TABLE
# ============================================================

PROFILE_DF = pd.DataFrame(

    PROFILE_MATRIX,

    index=APPLIANCES,

    columns=PROFILE_FEATURE_NAMES

)

print("\n")
print("=" * 80)
print("        APPLIANCE PROFILE KNOWLEDGE")
print("=" * 80)

display(PROFILE_DF)


print("\nPROFILE_MATRIX shape:",
      PROFILE_MATRIX.shape)

print(
    "Expected shape:",
    (len(APPLIANCES), 12)
)

Number of profile features: 12
['typical_power_on', 'delta_power_on_off', 'standby_power', 'power_std_on', 'typical_on_duration_sec', 'typical_off_duration_sec', 'dominant_cycle_period_sec', 'duty_cycle', 'operations_per_day', 'peak_hour_of_day', 'hour_concentration', 'weekday_weekend_ratio']

Estimated sampling interval: 6.0 seconds

Training data ends at original row: 6628331


        APPLIANCE PROFILE KNOWLEDGE


,typical_power_on,delta_power_on_off,standby_power,power_std_on,typical_on_duration_sec,typical_off_duration_sec,dominant_cycle_period_sec,duty_cycle,operations_per_day,peak_hour_of_day,hour_concentration,weekday_weekend_ratio
laptop,19.0,19.0,0.0,8.989161,42.0,6.0,60.0,0.196272,23.633802,9.0,0.333454,2.002343
washing_machine,211.0,211.0,0.0,727.945129,24.0,6.0,30.0,0.041998,73.331993,14.0,0.114137,0.861705
tv,100.0,99.0,1.0,19.277967,54.0,6.0,60.0,0.089437,66.563377,20.0,0.532732,0.858917
fridge,88.0,88.0,0.0,40.783360,48.0,6.0,66.0,0.375414,325.267609,18.0,0.451317,0.977809
microwave,1558.0,1557.0,1.0,617.539001,30.0,960.0,990.0,0.002935,5.983903,17.0,0.012534,1.167600



PROFILE_MATRIX shape: (5, 12)
Expected shape: (5, 12)


In [ ]:
#  ============================================================
# CELL 11 — TRAIN / RESUME 5 APPLIANCE-SPECIFIC SPECIALISTS
# ============================================================

specialist_models = {}
feature_models = {}

for j, app in enumerate(APPLIANCES):

    print("\n")
    print("=" * 80)
    print(f"        {app.upper()} SPECIALIST MODEL")
    print("=" * 80)

    # --------------------------------------------------------
    # Path where this appliance model will be saved
    # --------------------------------------------------------

    model_path = os.path.join(
        BASE_DIR,
        f"{app}_specialist.keras"
    )

    print("Model path:")
    print(model_path)

    # --------------------------------------------------------
    # GET THIS APPLIANCE'S 12 PROFILE FEATURES
    # --------------------------------------------------------
    #
    # PROFILE_MATRIX shape:
    #     (5 appliances, 12 features)
    #
    # PROFILE_MATRIX[j] gives the profile for the current
    # appliance.
    #
    # Example:
    #     j = 0 → Laptop profile
    #     j = 1 → Washing Machine profile
    #     j = 2 → TV profile
    #     j = 3 → Fridge profile
    #     j = 4 → Microwave profile
    #
    # The same appliance profile is repeated for every window.
    # --------------------------------------------------------

    profile_train = np.repeat(
        PROFILE_MATRIX[j][None, :],
        len(X_train_s),
        axis=0
    ).astype(np.float32)

    profile_meta = np.repeat(
        PROFILE_MATRIX[j][None, :],
        len(X_meta_s),
        axis=0
    ).astype(np.float32)

    print("\nProfile input:")
    print("Profile shape:", profile_train.shape)

    # --------------------------------------------------------
    # GET ONLY THIS APPLIANCE'S LABEL
    # --------------------------------------------------------
    #
    # Example:
    #
    # Laptop specialist:
    #     Y_train[:, 0]
    #
    # Washing machine specialist:
    #     Y_train[:, 1]
    #
    # etc.
    # --------------------------------------------------------

    y_train_app = Y_train[:, j]
    y_meta_app = Y_meta[:, j]

    print("\nTraining target:")
    print("Appliance:", app)
    print("Train labels:", y_train_app.shape)
    print("Meta labels :", y_meta_app.shape)

    print(
        "Train ON:",
        int(np.sum(y_train_app == 1)),
        "| Train OFF:",
        int(np.sum(y_train_app == 0))
    )

    print(
        "Meta ON:",
        int(np.sum(y_meta_app == 1)),
        "| Meta OFF:",
        int(np.sum(y_meta_app == 0))
    )

    # ========================================================
    # CHECK WHETHER THIS MODEL WAS ALREADY SAVED
    # ========================================================

    if os.path.exists(model_path):

        print("\n")
        print(">>> SAVED MODEL FOUND")
        print(f">>> Loading {app} specialist...")

        model = load_model(model_path)

        print(f">>> {app} specialist loaded successfully.")

    else:

        print("\n")
        print(">>> NO SAVED MODEL FOUND")
        print(f">>> Training {app} specialist from scratch...")

        # ----------------------------------------------------
        # BUILD SPECIALIST MODEL
        # ----------------------------------------------------

        model, feature_model_temp = build_specialist_model()

        print("\nModel created.")
        print(f"Training {app} specialist...")

        # ----------------------------------------------------
        # CHECKPOINT
        # ----------------------------------------------------
        #
        # This saves the BEST validation-loss model.
        #
        # Therefore, after every epoch:
        #
        #     if current val_loss < previous best
        #             ↓
        #        save model
        #
        # The file is:
        #
        #     laptop_specialist.keras
        #     washing_machine_specialist.keras
        #     etc.
        # ----------------------------------------------------

        checkpoint = ModelCheckpoint(
            filepath=model_path,
            monitor="val_loss",
            mode="min",
            save_best_only=True,
            save_weights_only=False,
            verbose=1
        )

        # ----------------------------------------------------
        # TRAIN MODEL
        # ----------------------------------------------------

        history = model.fit(

            # Input 1:
            # Aggregate household power
            [
                X_train_s[..., None],

                # Input 2:
                # Appliance-specific 12-feature profile
                profile_train
            ],

            # Target:
            # ONLY current appliance
            y_train_app,

            # Validation data
            validation_data=(
                [
                    X_meta_s[..., None],
                    profile_meta
                ],
                y_meta_app
            ),

            epochs=SPECIALIST_EPOCHS,

            batch_size=BATCH_SIZE,

            verbose=1,

            callbacks=[
                checkpoint
            ]
        )

        print("\n")
        print("=" * 80)
        print(f"{app.upper()} TRAINING COMPLETED")
        print("=" * 80)

        print("\nBest model saved at:")
        print(model_path)

        # ----------------------------------------------------
        # LOAD THE BEST SAVED MODEL
        # ----------------------------------------------------
        #
        # Important:
        # The final epoch isn't necessarily the best epoch.
        #
        # We therefore load the best checkpoint.
        # ----------------------------------------------------

        model = load_model(model_path)

        print(f"Best {app} model loaded successfully.")

    # ========================================================
    # STORE THE SPECIALIST MODEL
    # ========================================================

    specialist_models[app] = model

    # ========================================================
    # CREATE 128-D FEATURE EXTRACTOR
    # ========================================================
    #
    # The specialist model has:
    #
    # Aggregate
    #      ↓
    # CNN
    #      ↓
    # BiLSTM
    #      ↓
    # Knowledge/profile branch
    #      ↓
    # Fusion
    #      ↓
    # learned_feature_128
    #      ↓
    # Probability
    #
    # We extract the 128-D feature for the Meta Model.
    # ========================================================

    feature_models[app] = Model(
        inputs=model.inputs,
        outputs=model.get_layer(
            "learned_feature_128"
        ).output
    )

    print("\n128-D feature extractor created.")

    # ========================================================
    # VERIFY MODEL FILE
    # ========================================================

    if os.path.exists(model_path):

        file_size_mb = os.path.getsize(model_path) / (1024 * 1024)

        print(
            f"Saved file verified: {file_size_mb:.2f} MB"
        )

    else:

        print("WARNING: Model file was not found!")

    print("\n")


# ============================================================
# FINAL CHECK — ALL 5 SPECIALISTS
# ============================================================

print("\n")
print("=" * 80)
print("              SPECIALIST MODEL SUMMARY")
print("=" * 80)

for app in APPLIANCES:

    model_path = os.path.join(
        BASE_DIR,
        f"{app}_specialist.keras"
    )

    if os.path.exists(model_path):
        status = "SAVED ✓"
    else:
        status = "NOT FOUND ✗"

    print(
        f"{app:20s} → {status}"
    )

print("\n")
print("=" * 80)
print("       ALL AVAILABLE SPECIALIST MODELS ARE READY")
print("=" * 80)

print("\nModels available in memory:")
for app in specialist_models:
    print(f"  ✓ {app}")

print("\nFeature extractors available:")
for app in feature_models:
    print(f"  ✓ {app} → 128-D feature")



        LAPTOP SPECIALIST MODEL
Model path:
/content/gdrive/MyDrive/NILM_Approach2_2Years/improved_100_center/specialist_models/laptop_specialist.keras

Profile input:
Profile shape: (331415, 12)

Training target:
Appliance: laptop
Train labels: (331415,)
Meta labels : (71017,)
Train ON: 65003 | Train OFF: 266412
Meta ON: 9043 | Meta OFF: 61974


>>> SAVED MODEL FOUND
>>> Loading laptop specialist...
>>> laptop specialist loaded successfully.

128-D feature extractor created.
Saved file verified: 1.45 MB




        WASHING_MACHINE SPECIALIST MODEL
Model path:
/content/gdrive/MyDrive/NILM_Approach2_2Years/improved_100_center/specialist_models/washing_machine_specialist.keras

Profile input:
Profile shape: (331415, 12)

Training target:
Appliance: washing_machine
Train labels: (331415,)
Meta labels : (71017,)
Train ON: 13906 | Train OFF: 317509
Meta ON: 3314 | Meta OFF: 67703


>>> SAVED MODEL FOUND
>>> Loading washing_machine specialist...
>>> washing_machine specialist loaded succes

In [ ]:
# Cell 12 — Check specialist performance on META set

for j, app in enumerate(APPLIANCES):
    profile_meta = np.repeat(
        PROFILE_MATRIX[j][None, :],
        len(X_meta_s),
        axis=0
    ).astype(np.float32)

    p = specialist_models[app].predict(
        [X_meta_s[..., None], profile_meta],
        batch_size=BATCH_SIZE,
        verbose=0
    ).ravel()

    pred = (p >= 0.5).astype(int)
    y = Y_meta[:, j].astype(int)

    print(
        f"{app:18s} "
        f"Acc={accuracy_score(y,pred):.4f} "
        f"Prec={precision_score(y,pred,zero_division=0):.4f} "
        f"Recall={recall_score(y,pred,zero_division=0):.4f} "
        f"F1={f1_score(y,pred,zero_division=0):.4f}"
    )

laptop             Acc=0.8904 Prec=0.6575 Recall=0.2906 F1=0.4031
washing_machine    Acc=0.9912 Prec=0.8494 Recall=0.9855 F1=0.9124
tv                 Acc=0.9449 Prec=0.7610 Recall=0.5807 F1=0.6587
fridge             Acc=0.8173 Prec=0.8154 Recall=0.7477 F1=0.7801
microwave          Acc=0.9965 Prec=0.8382 Recall=0.3373 F1=0.4810


In [ ]:
# Cell 13 — Build META INPUT
#
# Each specialist receives its own 12-feature appliance profile.
# Output:
#   5 × 128-D learned features = 640
#   5 specialist probabilities = 5
#   Total = 645

def make_specialist_stack(Xd):
    feature_list = []
    prob_list = []

    for j, app in enumerate(APPLIANCES):
        profile = np.repeat(
            PROFILE_MATRIX[j][None, :],
            len(Xd),
            axis=0
        ).astype(np.float32)

        p = specialist_models[app].predict(
            [Xd[..., None], profile],
            batch_size=BATCH_SIZE,
            verbose=0
        ).reshape(-1, 1)

        f = feature_models[app].predict(
            [Xd[..., None], profile],
            batch_size=BATCH_SIZE,
            verbose=0
        )

        prob_list.append(p)
        feature_list.append(f)

    features = np.concatenate(feature_list, axis=1)
    probs = np.concatenate(prob_list, axis=1)

    return np.concatenate([features, probs], axis=1).astype(np.float32)

META_INPUT = make_specialist_stack(X_meta_s)

print("META_INPUT shape:", META_INPUT.shape)
print("Expected:", (len(X_meta_s), 645))

META_INPUT shape: (71017, 645)
Expected: (71017, 645)


In [ ]:
# Cell 13 — Build Meta Model

def build_meta_model(input_dim=645):
    inp = Input(shape=(input_dim,), name="specialist_fusion_input")

    x = Dense(256, activation="relu")(inp)
    x = Dropout(0.30)(x)

    x = Dense(128, activation="relu")(x)
    x = Dropout(0.20)(x)

    out = Dense(5, activation="sigmoid", name="final_state_output")(x)

    model = Model(inp, out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model

META_MODEL_PATH = os.path.join(META_DIR, "approach2_meta_model.keras")

if os.path.exists(META_MODEL_PATH):
    meta_model = load_model(META_MODEL_PATH)
    print("Loaded saved meta model.")
else:
    meta_model = build_meta_model(645)
    meta_model.summary()

Model: "functional_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ specialist_fusion_input         │ (None, 645)            │             0 │
│ (InputLayer)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │       165,376 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ final_state_output (Dense)      │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 198,917 (777.02 KB)

 Trainable params: 198,917 (777.02 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Cell 14 — Train Meta Model

if not os.path.exists(META_MODEL_PATH):
    meta_model.fit(
        META_INPUT,
        Y_meta,
        epochs=META_EPOCHS,
        batch_size=BATCH_SIZE,
        validation_split=0.10,
        verbose=1,
        callbacks=[
            ModelCheckpoint(
                META_MODEL_PATH,
                monitor="val_loss",
                save_best_only=True,
                verbose=1
            )
        ]
    )
    meta_model = load_model(META_MODEL_PATH)
else:
    print("Meta model already trained; skipping retraining.")

Epoch 1/10
495/500 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.4543 - loss: 0.3276
Epoch 1: val_loss improved from None to 0.14942, saving model to /content/gdrive/MyDrive/NILM_Approach2_2Years/improved_100_center/meta_model/approach2_meta_model.keras

Epoch 1: finished saving model to /content/gdrive/MyDrive/NILM_Approach2_2Years/improved_100_center/meta_model/approach2_meta_model.keras
500/500 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.4997 - loss: 0.1935 - val_accuracy: 0.6587 - val_loss: 0.1494
Epoch 2/10
495/500 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.5215 - loss: 0.1472
Epoch 2: val_loss did not improve from 0.14942
500/500 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - accuracy: 0.5081 - loss: 0.1464 - val_accuracy: 0.5079 - val_loss: 0.1533
Epoch 3/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.4797 - loss: 0.1462
Epoch 3: val_loss improved from 0.14942 to 0.14364, saving model to /content/gdrive/MyDrive/NILM_Approach2_2Years/improved_100_center/meta_model/approa

In [ ]:
# Cell 17 — Build TEST INPUT

TEST_INPUT = make_specialist_stack(X_test_s)

print("TEST_INPUT shape:", TEST_INPUT.shape)
print("Expected:", (len(X_test_s), 645))

TEST_INPUT shape: (71018, 645)
Expected: (71018, 645)


In [ ]:
# ============================================================
# CELL 16 — FIND BEST THRESHOLD FOR EACH APPLIANCE
# ============================================================

from sklearn.metrics import f1_score
import numpy as np

# Get Meta Model probabilities on META set
META_PROB = meta_model.predict(
    META_INPUT,
    batch_size=BATCH_SIZE,
    verbose=1
)

print("META_PROB shape:", META_PROB.shape)

# Store best threshold for each appliance
BEST_THRESHOLDS = {}

print("\n")
print("=" * 70)
print("       FINDING BEST THRESHOLD FOR EACH APPLIANCE")
print("=" * 70)

# Try thresholds from 0.05 to 0.95
thresholds = np.arange(0.05, 0.96, 0.01)

for j, app in enumerate(APPLIANCES):

    y_true = Y_meta[:, j].astype(int)

    best_threshold = 0.50
    best_f1 = -1

    for threshold in thresholds:

        y_pred = (
            META_PROB[:, j] >= threshold
        ).astype(int)

        score = f1_score(
            y_true,
            y_pred,
            zero_division=0
        )

        if score > best_f1:

            best_f1 = score
            best_threshold = threshold

    BEST_THRESHOLDS[app] = best_threshold

    print(
        f"{app:20s} "
        f"Best Threshold = {best_threshold:.2f} "
        f"Meta F1 = {best_f1:.4f}"
    )

print("\n")
print("Best thresholds:")
print(BEST_THRESHOLDS)

555/555 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step
META_PROB shape: (71017, 5)


       FINDING BEST THRESHOLD FOR EACH APPLIANCE
laptop               Best Threshold = 0.32 Meta F1 = 0.6399
washing_machine      Best Threshold = 0.57 Meta F1 = 0.9120
tv                   Best Threshold = 0.39 Meta F1 = 0.7696
fridge               Best Threshold = 0.46 Meta F1 = 0.8333
microwave            Best Threshold = 0.37 Meta F1 = 0.6742


Best thresholds:
{'laptop': np.float64(0.32000000000000006), 'washing_machine': np.float64(0.5700000000000002), 'tv': np.float64(0.39000000000000007), 'fridge': np.float64(0.4600000000000001), 'microwave': np.float64(0.37000000000000005)}


In [ ]:
# ============================================================
# CELL 17 — FINAL TEST PREDICTIONS USING APPLIANCE THRESHOLDS
# ============================================================

# Get final probabilities on completely unseen TEST data
Y_prob = meta_model.predict(
    TEST_INPUT,
    batch_size=BATCH_SIZE,
    verbose=1
)

print("Y_prob shape:", Y_prob.shape)


# ------------------------------------------------------------
# Apply a DIFFERENT threshold for each appliance
# ------------------------------------------------------------

Y_pred = np.zeros_like(Y_prob, dtype=int)

for j, app in enumerate(APPLIANCES):

    threshold = BEST_THRESHOLDS[app]

    Y_pred[:, j] = (
        Y_prob[:, j] >= threshold
    ).astype(int)

    print(
        f"{app:20s} "
        f"Threshold = {threshold:.2f}"
    )


print("\nFinal prediction shape:", Y_pred.shape)

555/555 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step
Y_prob shape: (71018, 5)
laptop               Threshold = 0.32
washing_machine      Threshold = 0.57
tv                   Threshold = 0.39
fridge               Threshold = 0.46
microwave            Threshold = 0.37

Final prediction shape: (71018, 5)


In [ ]:
# ============================================================
# CELL 18 — FINAL EVALUATION
# ============================================================

rows = []

for j, app in enumerate(APPLIANCES):

    y = Y_test[:, j].astype(int)
    pred = Y_pred[:, j]

    try:
        auc = roc_auc_score(
            y,
            Y_prob[:, j]
        )
    except ValueError:
        auc = np.nan

    rows.append({

        "Appliance": app,

        "Accuracy": accuracy_score(
            y,
            pred
        ),

        "Precision": precision_score(
            y,
            pred,
            zero_division=0
        ),

        "Recall": recall_score(
            y,
            pred,
            zero_division=0
        ),

        "F1": f1_score(
            y,
            pred,
            zero_division=0
        ),

        "ROC_AUC": auc
    })


results = pd.DataFrame(rows)

display(results)

,Appliance,Accuracy,Precision,Recall,F1,ROC_AUC
0,laptop,0.794827,0.347436,0.458510,0.395319,0.794468
1,washing_machine,0.989214,0.837593,0.976065,0.901542,0.995441
2,tv,0.936706,0.619078,0.870209,0.723470,0.972518
3,fridge,0.759624,0.667581,0.920100,0.773759,0.863703
4,microwave,0.992706,0.714583,0.473757,0.569767,0.995112


## What changed

This version replaces the previous **13 generic window statistics** with your **12 appliance-profile knowledge features**.

Each specialist gets its own profile:

- Laptop specialist → laptop profile
- Washing-machine specialist → washing-machine profile
- TV specialist → TV profile
- Fridge specialist → fridge profile
- Microwave specialist → microwave profile

The profile is calculated using **training-period appliance data only**, then repeated for the windows belonging to that specialist. This avoids using future meta/test appliance information to construct the knowledge input.

The model still receives **aggregate household power**, not the appliance power channel.

The 100-sample window and center-point label remain unchanged.